# Comparing runs

The same figures as `bubble_analysis.ipynb` (bubble count, size, total area, ...) per train, with **several runs on one
plot**. Each run is a results folder exported by `bubble_inference.ipynb` (e.g. `results/r557`).

Every run shows a jump in the bubble count between two trains: the new population of bubbles made by the laser scan of
the area. The notebook finds that train in each run (the largest rise of the count from one train to the next, or set
`SCAN_TRAINS` by hand) and can **align the runs on it**: x = trains since the scan, 0 being the first train after it.
Runs start from very different bubble counts, so every metric can also be shown **relative to before the scan**
(divided by the run's mean over its trains before the scan).

| section | figure |
|---|---|
| 2 | the scan train found in each run |
| 3 | bubble count (per frame and relative) |
| 4 | bubble size: mean, median, Sauter r₃₂, largest |
| 5 | total bubble area and area covered |
| 6 | polydispersity, elongated bubbles, volume proxy |
| 7 | the jump at the scan, per run (table) |
| 8 | size distribution just before and just after the scan |

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2
import bubble_stats as bst

bst.style()

## CONFIG

In [ ]:
RESULTS = ["results/r557", "results/r560", "results/r569", "results/r574"]   # results folders to compare (max 8 runs)
REVIEWED_ONLY = False         # only frames ticked "Reviewed" in napari
IN_ANALYSIS_ONLY = True       # apply the exclusions chosen at export (inner_margin, drop_edge_bubbles, ...)
COMPUTE_COVERAGE = True       # area covered by bubbles (union of outlines; a few seconds per frame)
UM_PER_PX = 3.2

SCAN_TRAINS = {}              # train in which the laser-scan bubbles appear, e.g. {"r560": 3}; runs not given: found
MIN_JUMP = 1.3                # automatic search: the count must rise at least this much (x) from one train to the next
ALIGN_ON_SCAN = True          # x = trains since the scan (False: train number)

SAVE_FIGS = True
FIG_DIR = os.path.join("results", "compare")

def save(fig, name):
    if SAVE_FIGS:
        os.makedirs(FIG_DIR, exist_ok=True)
        fig.savefig(os.path.join(FIG_DIR, name + ".png"), dpi=200, bbox_inches="tight")

## 1. Load

Metrics are recomputed from each folder's `bubbles.csv`, then averaged over the frames of a train (if there are several, the error bars show their standard error).

In [ ]:
bubbles, m = bst.load_runs(RESULTS, reviewed_only=REVIEWED_ONLY, in_analysis_only=IN_ANALYSIS_ONLY,
                           compute_coverage=COMPUTE_COVERAGE, um_per_px=UM_PER_PX)
mt = bst.per_train(m)
COLORS = bst.run_colors(mt["run"].unique())     # one colour per run, the same in every figure
print(f"{mt['run'].nunique()} runs, {len(m)} frames, {len(bubbles)} bubbles")
display(mt.pivot(index="train", columns="run", values="n_bubbles"))

## 2. The laser scan in each run

`jump` = bubble count in the scan train / count in the train before. Check it, and set `SCAN_TRAINS` for any run where it is wrong.

In [ ]:
scan = bst.find_scan_trains(mt, SCAN_TRAINS, min_jump=MIN_JUMP)
A = bst.align_runs(mt, scan, align=ALIGN_ON_SCAN)                    # absolute values
R = bst.align_runs(mt, scan, align=ALIGN_ON_SCAN, normalise=True)    # relative to before the scan

## 3. Bubble count

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 4.5))
bst.plot_runs(A, "n_bubbles", "bubbles per frame", ax=ax[0], align=ALIGN_ON_SCAN, colors=COLORS, title="Bubble count")
bst.plot_runs(R, "n_bubbles", "bubble count", ax=ax[1], align=ALIGN_ON_SCAN, normalise=True, colors=COLORS,
              title="Bubble count, relative to before the scan")
fig.tight_layout()
save(fig, "count_runs")

## 4. Bubble size

Mean and median describe the typical bubble, the Sauter radius r₃₂ the large ones (which hold most of the gas). A drop at the scan is the new population of small bubbles.

In [ ]:
SIZE = [("r_mean_um", "mean radius [µm]"), ("r_median_um", "median radius [µm]"),
        ("r32_um", "Sauter radius r₃₂ [µm]"), ("r_max_um", "largest bubble radius [µm]")]
fig = bst.plot_runs_grid(A, SIZE, align=ALIGN_ON_SCAN, colors=COLORS, suptitle="Bubble size")
save(fig, "size_runs")
fig = bst.plot_runs_grid(R, SIZE, align=ALIGN_ON_SCAN, normalise=True, colors=COLORS,
                         suptitle="Bubble size, relative to before the scan")
save(fig, "size_runs_relative")

## 5. Total bubble area

**Total** adds up all bubble areas (overlaps counted fully), **covered** is the area under at least one bubble.

In [ ]:
AREA = [("area_total_um2", "total bubble area [mm²]", 1e-6)] + \
       ([("coverage_frac", "area covered [% of the field of view]", 100)] if COMPUTE_COVERAGE else [])
fig = bst.plot_runs_grid(A, AREA, align=ALIGN_ON_SCAN, colors=COLORS, suptitle="Total bubble area")
save(fig, "area_runs")
fig = bst.plot_runs_grid(R, [p[:2] for p in AREA], align=ALIGN_ON_SCAN, normalise=True, colors=COLORS,
                         suptitle="Total bubble area, relative to before the scan")
save(fig, "area_runs_relative")

## 6. Polydispersity, shape and volume

In [ ]:
OTHER = [("polydispersity", "polydispersity (std / mean radius)"),
         ("frac_noncircular", "non-circular bubbles [%] (aspect < 0.8)", 100),
         ("volume_um3", "volume proxy Σ 4/3πr³ [10⁶ µm³]", 1e-6), ("density_per_mm2", "bubbles per mm²")]
fig = bst.plot_runs_grid(A, OTHER, align=ALIGN_ON_SCAN, colors=COLORS)
save(fig, "other_runs")

## 7. The jump at the scan

Last train before the scan vs the scan train, per run.

In [ ]:
jump = bst.scan_jump_table(mt, scan)
display(jump.set_index("run").filter(like="ratio").round(2))
display(jump.set_index("run").round(3))

## 8. Size distribution before and after the scan

In [ ]:
fig = bst.plot_size_before_after(bubbles, scan)
save(fig, "size_before_after")

## Notes

* Runs with no clear jump (largest rise below `MIN_JUMP`) are left out of the aligned / relative figures; set their
  scan train in `SCAN_TRAINS`, or use `ALIGN_ON_SCAN = False` to plot every run against its train number.
* Relative values divide by the mean over the trains before the scan, so a run with only one train before the scan
  has a noisier baseline.
* To compare more than 8 runs, split them into groups (one notebook run per group).